# 03 — Offline Evaluation (Precision/Recall/NDCG)
Simple holdout evaluation on the sample dataset to verify metrics.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

In [ ]:
import pandas as pd
from pathlib import Path
from src.utils.config import DATA_DIR
from src.models.metrics import precision_at_k, recall_at_k, ndcg_at_k

df = pd.read_csv(DATA_DIR/'sample_interactions.csv')
df = df.sort_values('timestamp')
test_idx = df.groupby('user_id')['timestamp'].idxmax()
test = df.loc[test_idx]
train = df.drop(test_idx)
true_items = test.groupby('user_id')['item_id'].apply(list)
pop = train.groupby('item_id')['rating'].sum().sort_values(ascending=False).index.tolist()
pred = {u: [i for i in pop if i not in set(train[train.user_id==u].item_id)][:10] for u in true_items.index}
prec = sum(precision_at_k(true_items[u], pred[u], 10) for u in pred)/len(pred)
rec = sum(recall_at_k(true_items[u], pred[u], 10) for u in pred)/len(pred)
ndcg = sum(ndcg_at_k(true_items[u], pred[u], 10) for u in pred)/len(pred)
print({'precision@10': round(prec,4), 'recall@10': round(rec,4), 'ndcg@10': round(ndcg,4)})